<a href="https://colab.research.google.com/github/ashinsiji/ict_assignment/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

2. Student Performance Analyzer
Create a Student Performance Analyzer that stores academic records and derives meaningful insights. The system should allow adding students with roll numbers and names, entering marks for multiple subjects, and storing them in structured formats. It should calculate individual averages, subject-wise toppers, pass/fail status based on defined rules, and overall ranking without using built-in sorting functions. Marks must be validated, grading rules should differ across subjects, and summary reports must be generated using loops and conditional logic. The program must be fully menu-driven and modular

In [1]:
import os

# Global list to store student records
student_records = []

# Define subjects for consistency
SUBJECTS = ["Math", "Physics", "English"]

def clear_screen():
    """Helper to clear the console screen."""
    os.system('cls' if os.name == 'nt' else 'clear')

# --- 1. Data Entry Modules ---

def add_student():
    """Adds a new student with a unique roll number."""
    print("\n--- Add New Student ---")
    try:
        roll_no = int(input("Enter Roll Number: "))

        # Check for duplicate roll number
        for student in student_records:
            if student['roll'] == roll_no:
                print("Error: Roll Number already exists!")
                return

        name = input("Enter Student Name: ")

        # Initialize student structure
        new_student = {
            'roll': roll_no,
            'name': name,
            'marks': {},
            'total': 0,
            'average': 0.0,
            'grades': {},
            'status': 'Pending'
        }
        student_records.append(new_student)
        print(f"Student {name} added successfully.")

    except ValueError:
        print("Invalid input! Roll number must be an integer.")

def enter_marks():
    """Enters marks for a specific student with validation."""
    print("\n--- Enter Marks ---")
    if not student_records:
        print("No students found. Please add a student first.")
        return

    try:
        target_roll = int(input("Enter Roll Number to add marks for: "))
        student = None

        # Find student
        for s in student_records:
            if s['roll'] == target_roll:
                student = s
                break

        if not student:
            print("Student not found.")
            return

        print(f"Entering marks for {student['name']}:")
        for sub in SUBJECTS:
            while True:
                try:
                    mark = float(input(f"  {sub} (0-100): "))
                    if 0 <= mark <= 100:
                        student['marks'][sub] = mark
                        break
                    else:
                        print("  Error: Marks must be between 0 and 100.")
                except ValueError:
                    print("  Error: Please enter a valid number.")

        print("Marks updated successfully.")

    except ValueError:
        print("Invalid input.")

# --- 2. Calculation & Logic Modules ---

def calculate_grade(subject, mark):
    """
    Applies different grading rules based on the subject.
    Constraint: Grading rules differ across subjects.
    """
    if subject == "Math":
        # Math is strict: needs 90+ for A
        if mark >= 90: return 'A'
        elif mark >= 75: return 'B'
        elif mark >= 50: return 'C'
        else: return 'F'

    elif subject == "Physics":
        # Standard grading
        if mark >= 85: return 'A'
        elif mark >= 70: return 'B'
        elif mark >= 40: return 'C'
        else: return 'F'

    elif subject == "English":
        # English is lenient: 80+ is A
        if mark >= 80: return 'A'
        elif mark >= 60: return 'B'
        elif mark >= 35: return 'C'
        else: return 'F'

    return 'N/A'

def process_results():
    """Calculates totals, averages, grades, and pass/fail status."""
    for student in student_records:
        if not student['marks']:
            continue # Skip students with no marks

        total_marks = 0
        has_failed_subject = False

        for sub in SUBJECTS:
            mark = student['marks'].get(sub, 0)
            total_marks += mark

            # Calculate Grade
            student['grades'][sub] = calculate_grade(sub, mark)

            # Check individual subject pass criteria (e.g., < 35 is fail generally)
            if mark < 35:
                has_failed_subject = True

        avg = total_marks / len(SUBJECTS)
        student['total'] = total_marks
        student['average'] = round(avg, 2)

        # Determine Status
        if has_failed_subject:
            student['status'] = "Fail (Sub)"
        elif avg >= 40:
            student['status'] = "Pass"
        else:
            student['status'] = "Fail (Avg)"

def manual_sort_ranking():
    """
    Sorts students based on average score using Bubble Sort.
    Constraint: NO built-in sort functions allowed.
    """
    n = len(student_records)
    # We copy the list to avoid messing up the original order if needed,
    # but here we will sort the global list in place for the report.
    for i in range(n):
        for j in range(0, n - i - 1):
            # Sort Descending (Highest to Lowest)
            if student_records[j]['average'] < student_records[j+1]['average']:
                # Swap
                student_records[j], student_records[j+1] = student_records[j+1], student_records[j]

# --- 3. Reporting Modules ---

def display_subject_toppers():
    """Identifies and prints the topper for each subject."""
    print("\n--- Subject Toppers ---")
    if not student_records:
        print("No data available.")
        return

    for sub in SUBJECTS:
        highest_mark = -1
        top_student = None

        for s in student_records:
            mark = s['marks'].get(sub, 0)
            if mark > highest_mark:
                highest_mark = mark
                top_student = s['name']

        print(f"{sub:<10}: {top_student} ({highest_mark})")

def generate_report():
    """Generates the full rank list and summary."""
    process_results() # Update calculations first
    manual_sort_ranking() # Sort by rank

    print("\n" + "="*85)
    print(f"{'Rank':<6}{'Roll':<6}{'Name':<15}{'Math':<8}{'Phy':<8}{'Eng':<8}{'Total':<8}{'Avg':<8}{'Status':<10}")
    print("="*85)

    rank = 1
    for s in student_records:
        if not s['marks']: continue # Skip empty records

        m = s['marks']
        print(f"{rank:<6}{s['roll']:<6}{s['name']:<15}"
              f"{m.get('Math',0):<8}{m.get('Physics',0):<8}{m.get('English',0):<8}"
              f"{s['total']:<8}{s['average']:<8}{s['status']:<10}")
        rank += 1
    print("="*85)

    display_subject_toppers()

# --- 4. Main Driver ---

def main_menu():
    while True:
        print("\n--- STUDENT PERFORMANCE ANALYZER ---")
        print("1. Add Student")
        print("2. Enter/Update Marks")
        print("3. Generate Performance Report (Rank List)")
        print("4. Exit")

        choice = input("Enter your choice (1-4): ")

        if choice == '1':
            add_student()
        elif choice == '2':
            enter_marks()
        elif choice == '3':
            if not student_records:
                print("No records to process.")
            else:
                generate_report()
        elif choice == '4':
            print("Exiting system. Goodbye!")
            break
        else:
            print("Invalid choice. Please try again.")

if __name__ == "__main__":
    main_menu()


--- STUDENT PERFORMANCE ANALYZER ---
1. Add Student
2. Enter/Update Marks
3. Generate Performance Report (Rank List)
4. Exit
Enter your choice (1-4): 1

--- Add New Student ---
Enter Roll Number: 15
Enter Student Name: xyem
Student xyem added successfully.

--- STUDENT PERFORMANCE ANALYZER ---
1. Add Student
2. Enter/Update Marks
3. Generate Performance Report (Rank List)
4. Exit
Enter your choice (1-4): 2

--- Enter Marks ---
Enter Roll Number to add marks for: 15
Entering marks for xyem:
  Math (0-100): 55
  Physics (0-100): 48
  English (0-100): 74
Marks updated successfully.

--- STUDENT PERFORMANCE ANALYZER ---
1. Add Student
2. Enter/Update Marks
3. Generate Performance Report (Rank List)
4. Exit
Enter your choice (1-4): 3

Rank  Roll  Name           Math    Phy     Eng     Total   Avg     Status    
1     15    xyem           55.0    48.0    74.0    177.0   59.0    Pass      

--- Subject Toppers ---
Math      : xyem (55.0)
Physics   : xyem (48.0)
English   : xyem (74.0)

--- S

5. Hostel Room Allocation System
Build a Hostel Room Allocation and Management System. The program should manage rooms with defined capacities and student registrations. Students can be allocated rooms based on availability and constraints such as year or gender (optional). The system must ensure one student occupies only one room at a time. Vacating rooms, viewing occupancy status, and generating allocation reports should be supported.

In [2]:
import os

# Database simulation
rooms = {
    "101": {"capacity": 2, "occupants": [], "gender": "Male"},
    "102": {"capacity": 2, "occupants": [], "gender": "Male"},
    "201": {"capacity": 2, "occupants": [], "gender": "Female"},
    "202": {"capacity": 2, "occupants": [], "gender": "Female"},
}
students = []

def register_student():
    print("\n--- Student Registration ---")
    roll = input("Enter Roll Number: ")
    # Check if student already exists
    if any(s['roll'] == roll for s in students):
        print("Error: Student already registered.")
        return

    name = input("Enter Name: ")
    gender = input("Enter Gender (Male/Female): ").capitalize()
    students.append({"roll": roll, "name": name, "gender": gender, "room": None})
    print(f"Student {name} registered.")

def allocate_room():
    print("\n--- Allocate Room ---")
    roll = input("Enter Student Roll Number: ")
    student = next((s for s in students if s['roll'] == roll), None)

    if not student:
        print("Student not found. Register first.")
        return
    if student['room']:
        print(f"Student already in Room {student['room']}.")
        return

    room_no = input("Enter Room Number: ")
    if room_no in rooms:
        room = rooms[room_no]
        # Logic Constraints
        if len(room['occupants']) >= room['capacity']:
            print("Error: Room is full.")
        elif room['gender'] != student['gender']:
            print(f"Error: This is a {room['gender']} room.")
        else:
            room['occupants'].append(student['name'])
            student['room'] = room_no
            print(f"Successfully allocated Room {room_no} to {student['name']}.")
    else:
        print("Invalid Room Number.")

def vacate_room():
    roll = input("\nEnter Roll Number to vacate: ")
    student = next((s for s in students if s['roll'] == roll), None)

    if student and student['room']:
        room_no = student['room']
        rooms[room_no]['occupants'].remove(student['name'])
        student['room'] = None
        print(f"Student {student['name']} has vacated Room {room_no}.")
    else:
        print("Student is not currently in a room.")

def status_report():
    print("\n--- Hostel Occupancy Report ---")
    for r_no, details in rooms.items():
        occ = len(details['occupants'])
        cap = details['capacity']
        print(f"Room {r_no} [{details['gender']}]: {occ}/{cap} occupied. ({', '.join(details['occupants'])})")

def main():
    while True:
        print("\n--- HOSTEL MANAGEMENT SYSTEM ---")
        print("1. Register Student\n2. Allocate Room\n3. Vacate Room\n4. Status Report\n5. Exit")
        choice = input("Select Option: ")
        if choice == '1': register_student()
        elif choice == '2': allocate_room()
        elif choice == '3': vacate_room()
        elif choice == '4': status_report()
        elif choice == '5': break

if __name__ == "__main__":
    main()


--- HOSTEL MANAGEMENT SYSTEM ---
1. Register Student
2. Allocate Room
3. Vacate Room
4. Status Report
5. Exit
Select Option: 1

--- Student Registration ---
Enter Roll Number: 223
Enter Name: gautham
Enter Gender (Male/Female): male
Student gautham registered.

--- HOSTEL MANAGEMENT SYSTEM ---
1. Register Student
2. Allocate Room
3. Vacate Room
4. Status Report
5. Exit
Select Option: 2

--- Allocate Room ---
Enter Student Roll Number: 223
Enter Room Number: 4
Invalid Room Number.

--- HOSTEL MANAGEMENT SYSTEM ---
1. Register Student
2. Allocate Room
3. Vacate Room
4. Status Report
5. Exit
Select Option: 3

Enter Roll Number to vacate: 223
Student is not currently in a room.

--- HOSTEL MANAGEMENT SYSTEM ---
1. Register Student
2. Allocate Room
3. Vacate Room
4. Status Report
5. Exit
Select Option: 4

--- Hostel Occupancy Report ---
Room 101 [Male]: 0/2 occupied. ()
Room 102 [Male]: 0/2 occupied. ()
Room 201 [Female]: 0/2 occupied. ()
Room 202 [Female]: 0/2 occupied. ()

--- HOSTEL MAN